# Setup Enviroment

## Create Resource Group

using SDK Python

In [16]:
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient
from dotenv import load_dotenv
import os

# Load environment variables
load_dotenv()

# Initialize credentials and clients
credential = DefaultAzureCredential()
subscription_id = os.getenv("AZURE_SUBSCRIPTION_ID")
resource_group = os.getenv("AZURE_RESOURCE_GROUP")
location = os.getenv("AZURE_LOCATION")
hub_name = os.getenv("AZURE_AI_HUB_NAME")


In [18]:
project_name = os.getenv("AZURE_AI_PROJECT_NAME")

In [17]:

# Create resource group
resource_client = ResourceManagementClient(credential, subscription_id)
rg_result = resource_client.resource_groups.create_or_update(
    resource_group,
    {"location": location}
)
print(f"Created resource group: {rg_result.name}")


Created resource group: rg-rag-sdk


In [19]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Hub, Project

# Create Azure AI Foundry hub
ml_client = MLClient(credential, subscription_id, resource_group)
hub = Hub(
    name=hub_name,
    location=location,
    display_name=f"{hub_name}",
    resource_group=resource_group
)
created_hub = ml_client.workspaces.begin_create(hub).result()
print(f"Created hub: {created_hub.name}")


The deployment request rag-openai-hub-7751315 was accepted. ARM deployment URI for reference: 
https://portal.azure.com//#blade/HubsExtension/DeploymentDetailsBlade/overview/id/%2Fsubscriptions%2F562be4bd-42dc-4fb6-8e07-e3fd5400b9f7%2FresourceGroups%2Frg-rag-sdk%2Fproviders%2FMicrosoft.Resources%2Fdeployments%2Frag-openai-hub-7751315
Creating Storage Account: (ragopenastorage40eb6d2c8  ) ...  Done (24s)
Creating Key Vault: (ragopenakeyvault9d9aba26  )  Done (18s)
Creating AzureML Workspace: (rag-openai-hub  ) ....  Done (38s)
Total time : 1m 9s



Created hub: rag-openai-hub


In [ ]:

# Create Azure AI Services resource for OpenAI
cog_client = CognitiveServicesManagementClient(credential, subscription_id)
ai_services_name = f"{hub_name}-aiservices"
ai_services_params = {
    "location": location,
    "kind": "OpenAI",
    "sku": {"name": "S0"},
    "properties": {}
}
ai_services_result = cog_client.accounts.begin_create(
    resource_group_name=resource_group,
    account_name=ai_services_name,
    account=ai_services_params
).result()
print(f"Created AI Services resource: {ai_services_result.name}")


In [ ]:

# Create hub-based project
project = Project(
    name=project_name,
    location=location,
    display_name=f"{project_name} Display",
    resource_group=resource_group,
    hub_id=created_hub.id
)
created_project = ml_client.workspaces.begin_create(project).result()
print(f"Created project: {created_project.name}")


In [ ]:

# Save hub endpoint and key to .env
ai_services_key = cog_client.accounts.list_keys(resource_group, ai_services_name).key1
with open(".env", "w") as f:
    f.write(f"OPEN_AI_ENDPOINT=https://{ai_services_name}.openai.azure.com/\n")
    f.write(f"OPEN_AI_KEY={ai_services_key}\n")
    f.write(f"CHAT_MODEL={os.getenv('CHAT_MODEL')}\n")
    f.write(f"EMBEDDING_MODEL={os.getenv('EMBEDDING_MODEL')}\n")
    f.write(f"SEARCH_ENDPOINT={os.getenv('SEARCH_ENDPOINT')}\n")
    f.write(f"SEARCH_KEY={os.getenv('SEARCH_KEY')}\n")
    f.write(f"INDEX_NAME={os.getenv('INDEX_NAME')}\n")
    f.write(f"AZURE_SUBSCRIPTION_ID={subscription_id}\n")
    f.write(f"AZURE_RESOURCE_GROUP={resource_group}\n")
    f.write(f"AZURE_LOCATION={location}\n")
    f.write(f"AZURE_AI_HUB_NAME={hub_name}\n")
    f.write(f"AZURE_AI_PROJECT_NAME={project_name}\n")
print("Updated .env with hub endpoint and key")

using CLI Azure

In [ ]:
az group create \
  --name rg-rag-sdk \
  --location eastus2

In [ ]:
az ml workspace create \
  --name ai-hub-rag-sdk \
  --resource-group rg-rag-sdk \
  --location eastus \
  --kind hub
  --display-name "${AZURE_AI_HUB_NAME} Display"

## Retrieve the list of resource groups

using SDK Python

In [4]:
# Retrieve the list of resource groups
group_list = resource_client.resource_groups.list()

# Show the groups in formatted output
column_width = 40

print("Resource Group".ljust(column_width) + "Location")
print("-" * (column_width * 2))

for group in list(group_list):
    print(f"{group.name:<{column_width}}{group.location}")

Resource Group                          Location
--------------------------------------------------------------------------------
rg-rag-sdk                              eastus


List resources within a specific resource group

In [5]:
# Import the needed credential and management objects from the libraries.
from azure.identity import DefaultAzureCredential
from azure.mgmt.resource import ResourceManagementClient
from dotenv import load_dotenv
import os

load_dotenv()
# Acquire a credential object.
credential = DefaultAzureCredential()

# Retrieve subscription ID from environment variable.
subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]

# Retrieve the resource group to use, defaulting to "myResourceGroup".
resource_group = os.getenv("AZURE_RESOURCE_GROUP")

# Obtain the management object for resources.
resource_client = ResourceManagementClient(credential, subscription_id)

# Retrieve the list of resources in "myResourceGroup" (change to any name desired).
# The expand argument includes additional properties in the output.
resource_list = resource_client.resources.list_by_resource_group(
    resource_group, expand = "createdTime,changedTime")

# Show the groups in formatted output
column_width = 36

print("Resource".ljust(column_width) + "Type".ljust(column_width)
    + "Create date".ljust(column_width) + "Change date".ljust(column_width))
print("-" * (column_width * 4))

for resource in list(resource_list):
    print(f"{resource.name:<{column_width}}{resource.type:<{column_width}}"
       f"{str(resource.created_time):<{column_width}}{str(resource.changed_time):<{column_width}}")

Resource                            Type                                Create date                         Change date                         
------------------------------------------------------------------------------------------------------------------------------------------------
rag-openai-gpt41                    Microsoft.CognitiveServices/accounts2025-08-27 04:12:54.640469+00:00    2025-08-27 04:23:13.806361+00:00    


## Create Foundry Resource

SDK Python

In [7]:
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.mgmt.cognitiveservices import CognitiveServicesManagementClient

load_dotenv()
subscription_id = os.environ["AZURE_SUBSCRIPTION_ID"]
resource_group = os.environ["AZURE_RESOURCE_GROUP"]
foundry_project_name = os.environ["AZURE_FOUNDRY_PROJECT"]
foundry_resource_chatmodel = os.environ["AZURE_FOUNDRY_CHATMODEL"]
foundry_resource_embedding = os.environ["AZURE_FOUNDRY_EMBEDDING"]
location = os.environ["LOCATION"]

client = CognitiveServicesManagementClient(
    credential=DefaultAzureCredential(), 
    subscription_id=subscription_id,
    api_version="2025-04-01-preview"
)

credential=DefaultAzureCredential()

In [ ]:
# Depuración: imprime los valores usados y verifica acceso a la suscripción
print('subscription_id:', subscription_id)
print('resource_group:', resource_group)
print('foundry_resource_chatmodel:', foundry_resource_chatmodel)
print('foundry_project_name:', foundry_project_name)

# Verifica que el grupo de recursos existe usando Azure CLI desde Python
import subprocess
result = subprocess.run([
    'az', 'group', 'show', '--name', resource_group, '--subscription', subscription_id, '--output', 'json'
], capture_output=True, text=True)
if result.returncode == 0:
    print('El grupo de recursos existe y es accesible.')
else:
    print('No se encontró el grupo de recursos o no tienes acceso.')
    print(result.stderr)

CLI Azure

create project

In [ ]:
az cognitiveservices account create \
  --name rag-openai-gpt41 \
  --resource-group rg-rag-sdk \
  --kind OpenAI \
  --sku s0 \
  --location eastus

Model Avalible

In [ ]:
# Listar modelos disponibles con detalles relevantes (usando jq para formateo)
az cognitiveservices account list-models \
  -n rag-openai-gpt41 \
  -g rg-rag-sdk | \
  jq '.[] | { 
    name: .name, 
    format: .format, 
    version: .version, 
    sku: .skus[0].name, 
    capacity: .skus[0].capacity.default 
  }'

output example

In [ ]:
{
    "name": "Phi-3.5-vision-instruct",
    "format": "Microsoft",
    "version": "2",
    "sku": "GlobalStandard",
    "capacity": 1
}

### Creating Resources

deployment chat model: gpt-4.1

In [ ]:
az cognitiveservices account deployment create \
    -g rg-rag-sdk \
    -n rag-openai-gpt41 \
    --deployment-name gpt41-deployment-chat \
    --model-name gpt-4.1 \
    --model-version "2025-04-14" \
    --model-format OpenAI \
    --sku-capacity 1 \
    --sku-name GlobalStandard

deployment embedding model: text-embedding-ada-002

In [ ]:
az cognitiveservices account deployment create \
    -g rg-rag-sdk \
    -n rag-openai-gpt41 \
    --deployment-name ada-embedding-deployment \
    --model-name text-embedding-ada-002 \
    --model-version "2" \
    --model-format OpenAI \
    --sku-capacity 1 \
    --sku-name Standard

## Get endpoint

SDK Python

In [14]:
print(account.properties.endpoints)

{'OpenAI Language Model Instance API': 'https://eastus.api.cognitive.microsoft.com/', 'OpenAI Dall-E API': 'https://eastus.api.cognitive.microsoft.com/', 'OpenAI Sora API': 'https://eastus.api.cognitive.microsoft.com/', 'OpenAI Moderations API': 'https://eastus.api.cognitive.microsoft.com/', 'OpenAI Whisper API': 'https://eastus.api.cognitive.microsoft.com/', 'OpenAI Model Scaleset API': 'https://eastus.api.cognitive.microsoft.com/', 'OpenAI Realtime API': 'https://eastus.api.cognitive.microsoft.com/', 'Token Service API': 'https://eastus.api.cognitive.microsoft.com/'}


CLI Azure

In [ ]:
az cognitiveservices account show -n rag-openai-gpt41 -g rg-rag-sdk | jq '.properties.endpoints'

## Get apikey

SDK Python

In [15]:
print(client.accounts.list_keys(resource_group, "rag-openai-gpt41" ))

{'additional_properties': {}, 'key1': 'c567049315ae4ddbae04b3604196df46', 'key2': '6a96d7e2aed64f83b674fcc4f3318a3a'}


CLI Azure

In [ ]:
az cognitiveservices account keys list -n rag-openai-gpt41 -g rg-rag-sdk